# <b>SN-AI Project</b>

## <b>Project Objective</b>

Given a dataset containing all the **Serie A match schedules and results** from the **2017/2018 season to the 2024/2025 season**, the goal of this project is to build a **neural network model capable of predicting the results of Serie A matches for the 2025/2026 season**.

The model will be trained using historical match data in **JSON format**, which contains information such as:

- Home team  
- Away team  
- Goals scored by the home team  
- Goals scored by the away team  
- Match date  
- Stadium where the match was played  

### Example of a match entry in the dataset

```json
{
  "MatchNumber": 1,
  "RoundNumber": 1,
  "DateUtc": "2017-08-19 16:00:00Z",
  "Location": "Juventus Stadium",
  "HomeTeam": "Juventus",
  "AwayTeam": "Cagliari",
  "Group": null,
  "HomeTeamScore": 3,
  "AwayTeamScore": 0
}
```

### Expected Output
The expected outputs are the results of the matches in the Serie A 2025/26 schedule. Our goal is to compare them with the actual match results and see how many times it predicts correctly.

**Premise:** we know that real results are not based only on past results; there are many other factors that can influence them, such as the physical condition of the players on the field, yearly squad changes, and other factors.



In [29]:
# Reading files from json_files folder and reading their content

import os

# Function to read the content of a file
def read_file(file_path):
    print(f"Reading file: {file_path}")
    content = ""
    with open(file_path, "r") as file:
        content = file.read()
    return content

# Define the path to the folder containing the JSON files
train_path = "/Users/brunovincenzi/Coding/Università/MLSettiProgetto/json_files/train"
eval_path = "/Users/brunovincenzi/Coding/Università/MLSettiProgetto/json_files/eval"

# List all files in the folder
train_files = [
    f for f in os.listdir(train_path) if os.path.isfile(os.path.join(train_path, f))
]

eval_files = [
    f for f in os.listdir(eval_path) if os.path.isfile(os.path.join(eval_path, f))
]

train_file_contents = {}
eval_file_contents = {}

# Read the content of each file and store it in a dictionary
for file in train_files:
    file_path = os.path.join(train_path, file)
    content = read_file(file_path)
    train_file_contents[file] = content

for file in eval_files:
    file_path = os.path.join(eval_path, file)
    content = read_file(file_path)
    eval_file_contents[file] = content


Reading file: /Users/brunovincenzi/Coding/Università/MLSettiProgetto/json_files/train/SerieA_1920.json
Reading file: /Users/brunovincenzi/Coding/Università/MLSettiProgetto/json_files/train/SerieA_2223.json
Reading file: /Users/brunovincenzi/Coding/Università/MLSettiProgetto/json_files/train/SerieA_1819.json
Reading file: /Users/brunovincenzi/Coding/Università/MLSettiProgetto/json_files/train/SerieA_2324.json
Reading file: /Users/brunovincenzi/Coding/Università/MLSettiProgetto/json_files/train/SerieA_2021.json
Reading file: /Users/brunovincenzi/Coding/Università/MLSettiProgetto/json_files/train/SerieA_2122.json
Reading file: /Users/brunovincenzi/Coding/Università/MLSettiProgetto/json_files/train/SerieA_1718.json
Reading file: /Users/brunovincenzi/Coding/Università/MLSettiProgetto/json_files/eval/SerieA_2526.json
Reading file: /Users/brunovincenzi/Coding/Università/MLSettiProgetto/json_files/eval/SerieA_2425.json


In [30]:
# Parse the content of each file and saving it in a dictionary excluding unnecessary information
import json

# Dictionary to store all possible matches, example key: "Juventus-Inter" value: [  {"MatchNumber": 1, "RoundNumber": 1, "HomeTeam": "Juventus", "AwayTeam": "Inter", "HomeTeamScore": 3,"AwayTeamScore": 0}]
all_possible_past_matches = {}
all_possible_teams = {}

# Function to parse JSON content and extract matches
def parse_json(json_content):
    try:
        matches = json.loads(json_content)
        if not isinstance(matches, list):
            print("Expected a list of matches")
            return []

        for match in matches:
            # Remove keys safely
            match.pop("Location", None)
            match.pop("Group", None)
            match.pop("DateUtc", None)

            # Initialize team scores as integers
            if match["HomeTeam"] not in all_possible_teams:
                all_possible_teams[match["HomeTeam"]] = 0
            if match["AwayTeam"] not in all_possible_teams:
                all_possible_teams[match["AwayTeam"]] = 0
            
            if match["HomeTeamScore"] is not None:
                # Update scores
                if match["HomeTeamScore"] > match["AwayTeamScore"]:
                    all_possible_teams[match["HomeTeam"]] += 3
                elif match["HomeTeamScore"] < match["AwayTeamScore"]:
                    all_possible_teams[match["AwayTeam"]] += 3
                else:
                    all_possible_teams[match["HomeTeam"]] += 1
                    all_possible_teams[match["AwayTeam"]] += 1

                # Track past matches
                match_key = f"{match['HomeTeam']}-{match['AwayTeam']}"
                if match_key not in all_possible_past_matches:
                    all_possible_past_matches[match_key] = []

                m = match.copy()
                m.pop("HomeTeam", None)
                m.pop("AwayTeam", None)
                m.pop("MatchNumber", None)
                m.pop("RoundNumber", None)

                result = f"{m['HomeTeamScore']}-{m['AwayTeamScore']}"
                all_possible_past_matches[match_key].append(result)
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON: {e}")
        return []

    return matches
    
train_parsed_contents = {}

for file, content in train_file_contents.items():
    parsed_content = parse_json(content)
    train_parsed_contents[file] = parsed_content

print("All possible past matches:")
for match_key, results in all_possible_past_matches.items():
    print(f"{match_key}: {results}")

print("\nAll possible teams and their points:")
for team, points in all_possible_teams.items():
    print(f"{team}: {points} points")

eval_parsed_contents = {}
for file, content in eval_file_contents.items():
    parsed_content = parse_json(content)
    eval_parsed_contents[file] = parsed_content

All possible past matches:
Fiorentina-Torino: ['1-0', '0-1', '1-1', '1-1', '1-0', '2-1', '3-0']
Hellas Verona-Roma: ['0-0', '1-3', '3-2', '0-0', '3-2', '0-1']
Parma-Napoli: ['0-2', '0-4', '0-0', '0-2']
Genoa-Crotone: ['4-1', '4-1', '1-0']
Sassuolo-Cagliari: ['1-1', '3-0', '1-1', '2-2', '0-0']
Juventus-Sampdoria: ['3-0', '4-2', '2-1', '3-0', '3-2', '3-0']
Milan-Bologna: ['2-0', '2-0', '2-1', '3-1', '2-0', '0-0']
Torino-Atalanta: ['2-4', '1-2', '2-0', '2-1', '2-4', '1-2', '1-1']
Cagliari-Lazio: ['0-2', '1-2', '1-2', '0-2', '0-3', '2-2']
Sampdoria-Benevento: ['2-3', '2-3', '2-1']
Inter-Fiorentina: ['4-3', '0-1', '2-1', '2-1', '4-3', '1-1', '3-0']
Spezia-Sassuolo: ['1-4', '2-2', '1-4', '2-2']
Hellas Verona-Udinese: ['1-0', '1-2', '0-0', '1-0', '4-0', '0-1']
Napoli-Genoa: ['6-0', '1-1', '2-2', '6-0', '3-0', '1-0']
Crotone-Milan: ['0-2', '0-2']
Roma-Juventus: ['2-2', '1-0', '2-0', '1-1', '2-2', '3-4', '0-0']
Bologna-Parma: ['4-1', '4-1', '0-0', '4-1']
Benevento-Inter: ['2-5', '2-5', '1-2']
U